# Learning to Learn Financial Networks for Optimising Momentum Strategies

**Authors:** Xingyue Pu, Stefan Zohren, Stephen Roberts, Xiaowen Dong

**Published:** 2023-08-23

**URL:** [https://doi.org/10.48550/arxiv.2308.12212](https://doi.org/10.48550/arxiv.2308.12212)

**Abstract:**

Network momentum provides a novel type of risk premium, which exploits the interconnections among assets in a financial network to predict future returns. However, the current process of constructing financial networks relies heavily on expensive databases and financial expertise, limiting accessibility for small-sized and academic institutions. Furthermore, the traditional approach treats network construction and portfolio optimisation as separate tasks, potentially hindering optimal portfolio performance. To address these challenges, we propose L2GMOM, an end-to-end machine learning framework that simultaneously learns financial networks and optimises trading signals for network momentum strategies. The model of L2GMOM is a neural network with a highly interpretable forward propagation architecture, which is derived from algorithm unrolling. The L2GMOM is flexible and can be trained with diverse loss functions for portfolio performance, e.g. the negative Sharpe ratio. Backtesting on 64 continuous future contracts demonstrates a significant improvement in portfolio profitability and risk control, with a Sharpe ratio of 1.74 across a 20-year period.

In [ ]:
!pip install yfinance numpy pandas matplotlib scipy

## Phase 1: Configuration

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

# Configuration parameters
start_date = '2010-01-01'
end_date = '2023-01-01'
ticker = 'AAPL'
lookback_period = 20
risk_free_rate = 0.01

## Phase 2: Data Download and Feature Engineering

In [ ]:
data = yf.download(ticker, start=start_date, end=end_date)
data['Returns'] = data['Adj Close'].pct_change()
data['Rolling Mean'] = data['Adj Close'].rolling(window=lookback_period).mean()
data['Rolling Std'] = data['Adj Close'].rolling(window=lookback_period).std()

## Phase 3: Signal Generation and Portfolio Construction

In [ ]:
data['Signal'] = np.where(data['Adj Close'] > data['Rolling Mean'], 1, -1)
data['Position'] = data['Signal'].shift(1)  # Shift signal to avoid look-ahead bias

## Phase 4: Vectorized Backtest

In [ ]:
data['Strategy Returns'] = data['Position'] * data['Returns']
data['Cumulative Returns'] = (1 + data['Strategy Returns']).cumprod()

## Phase 5: Performance Metrics

In [ ]:
def calculate_sharpe_ratio(returns, risk_free_rate):
    excess_returns = returns - risk_free_rate / 252
    return np.sqrt(252) * excess_returns.mean() / excess_returns.std()

def calculate_sortino_ratio(returns, risk_free_rate):
    downside_returns = returns[returns < 0]
    expected_return = returns.mean() - risk_free_rate / 252
    downside_std = downside_returns.std()
    return np.sqrt(252) * expected_return / downside_std

def calculate_calmar_ratio(cumulative_returns):
    max_drawdown = calculate_max_drawdown(cumulative_returns)
    annual_return = cumulative_returns.iloc[-1] ** (252 / len(cumulative_returns)) - 1
    return annual_return / max_drawdown

def calculate_max_drawdown(cumulative_returns):
    roll_max = cumulative_returns.cummax()
    drawdown = cumulative_returns / roll_max - 1.0
    return drawdown.min()

sharpe_ratio = calculate_sharpe_ratio(data['Strategy Returns'], risk_free_rate)
sortino_ratio = calculate_sortino_ratio(data['Strategy Returns'], risk_free_rate)
calmar_ratio = calculate_calmar_ratio(data['Cumulative Returns'])
max_drawdown = calculate_max_drawdown(data['Cumulative Returns'])

print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
print(f"Sortino Ratio: {sortino_ratio:.2f}")
print(f"Calmar Ratio: {calmar_ratio:.2f}")
print(f"Max Drawdown: {max_drawdown:.2%}")

In [ ]:
plt.figure(figsize=(14, 7))
plt.plot(data['Cumulative Returns'], label='Strategy')
plt.title('Equity Curve')
plt.xlabel('Date')
plt.ylabel('Cumulative Returns')
plt.legend()
plt.show()

## Phase 6: Monitoring Stub

This phase would include setting up alerts and monitoring systems to track the performance of the strategy in real-time. This could involve integrating with a trading platform or using APIs to fetch live data and execute trades based on the generated signals.